
<font color = green >

## Home Task

</font>

Objective: Apply at least two (2) modern sentiment analysis methods to classify text data and evaluate their performance




Dataset:

Sentiment Analysis Dataset: https://www.cs.cornell.edu/people/pabo/movie-review-data/rt-polaritydata.tar.gz

alternative source:
[rt-polaritydata](https://github.com/dennybritz/cnn-text-classification-tf/tree/master/data/rt-polaritydata)

Each line in these two files corresponds to a single snippet (usually containing roughly one single sentence); all snippets are down-cased.

[More info about dataset](https://www.cs.cornell.edu/people/pabo/movie-review-data/rt-polaritydata.README.1.0.txt)


- rt-polarity.neg: Contains negative reviews.
- rt-polarity.pos: Contains positive reviews

### Task Description:

1. Data Loading & Preparation:

    - Load rt-polarity.neg and rt-polarity.pos.
    - Split each file into individual snippets.
    - Assign labels (0 for negative, 1 for positive).
    - Combine into a single dataset.
    - Split the dataset into training and testing sets.
2. Implement and evaluate at least three (3) methods from the lecture, prioritizing modern approaches.
3. For each implemented method, report and compare classification metrics: Accuracy, Precision, Recall, and F1-score.

## Import all libraries

In [118]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import(
    CountVectorizer,
    TfidfVectorizer
)
from sklearn.metrics import(
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset
import torch.nn.functional as F

## Path to reviews

In [119]:
cwd = os.getcwd()

path_to_neg_reviews = os.path.join(cwd, 'data', 'rt-polarity.neg')
path_to_pos_reviews = os.path.join(cwd, 'data', 'rt-polarity.pos')

## Reading data

In [120]:
fn = path_to_neg_reviews

with open(fn, "r",encoding='utf-8', errors='ignore') as f: # some invalid symbols encountered
    content = f.read()
texts_neg=  content.splitlines()
print ('len of texts_neg = {:,}'.format (len(texts_neg)))
for review in texts_neg[:5]:
    print ( '\n', review)

len of texts_neg = 5,331

 simplistic , silly and tedious . 

 it's so laddish and juvenile , only teenage boys could possibly find it funny . 

 exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

 [garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

 a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 


In [121]:
fn = path_to_pos_reviews

with open(fn, "r",encoding='utf-8', errors='ignore') as f:
    content = f.read()
texts_pos=  content.splitlines()
print ('len of texts_pos = {:,}'.format (len(texts_pos)))
for review in texts_pos[:5]:
    print ('\n', review)

len of texts_pos = 5,331

 the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

 the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

 effective but too-tepid biopic

 if you sometimes like to go to the movies to have fun , wasabi is a good place to start . 

 emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one . 


In [122]:
df_pos = pd.DataFrame(texts_pos, columns=['review'])
df_pos['status'] = 1

df_neg = pd.DataFrame(texts_neg, columns=['review'])
df_neg['status'] = 0

df_reviews = pd.concat([df_pos, df_neg], ignore_index=True)

In [123]:
df_reviews.sample(5, random_state=42)

,review,status
6830,"a dark , dull thriller with a parting shot tha...",0
8600,director chris eyre is going through the paces...,0
4080,"although it lacks the detail of the book , the...",1
3079,the script by david koepp is perfectly service...,1
582,"an exciting and involving rock music doc , a s...",1


## Split dataset to train and test sets

In [124]:
X_train, X_test, y_train, y_test = train_test_split(
    df_reviews['review'], 
    df_reviews['status'], 
    test_size=0.25, 
    random_state=20
)

## Bag-of-words

#### Extract Features

In [125]:
count_vect = CountVectorizer(stop_words='english').fit(X_train)

features = count_vect.get_feature_names_out()
print(f'Feature samples {features[::100]}\nSize: {len(features)}')

Feature samples ['00' '3000' 'absurdist' 'actresses' 'affect' 'alarmed' 'amir' 'antes'
 'archive' 'aspects' 'aunque' 'bacon' 'basically' 'believable' 'billy'
 'blobby' 'bombshell' 'brassy' 'bros' 'bushels' 'candor' 'cast' 'changes'
 'chick' 'circumstantial' 'cloak' 'college' 'compassion' 'condensed'
 'constitutes' 'convoluted' 'cowrote' 'cristo' 'cursory' 'david' 'def'
 'denial' 'desplechin' 'dig' 'discuss' 'disturbance' 'doors' 'drenched'
 'dutifully' 'effects' 'embrace' 'england' 'epochs' 'etre' 'executed'
 'expressionistic' 'falseness' 'feasting' 'filipino' 'flat' 'folly'
 'frankie' 'fugitive' 'garry' 'gibberish' 'gong' 'grass' 'grueling'
 'hammer' 'hatred' 'hemlock' 'hmmmmight' 'horrors' 'hybrid' 'imbued'
 'inappropriate' 'inequities' 'insanity' 'interaes' 'invites' 'jaglom'
 'joo' 'keg' 'korean' 'larry' 'legitimate' 'like' 'locations' 'lucy'
 'maintain' 'mario' 'mcbeal' 'menzel' 'millisecond' 'mnch' 'morbid'
 'multi' 'narcissistic' 'network' 'norris' 'obsession' 'open' 'outpaces'


In [126]:
X_train_vectorized = count_vect.transform(X_train)
X_test_vectorized = count_vect.transform(X_test)
X_train_vectorized

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 77239 stored elements and shape (7996, 15727)>

#### Review vectorized training sample

In [127]:
print(X_train_vectorized[0])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (1, 15727)>
  Coords	Values
  (0, 185)	1
  (0, 590)	1
  (0, 1018)	1
  (0, 1774)	1
  (0, 2470)	1
  (0, 2538)	1
  (0, 4533)	1
  (0, 5016)	1
  (0, 5307)	1
  (0, 5676)	1
  (0, 8053)	1
  (0, 9076)	1
  (0, 9852)	1
  (0, 10109)	1
  (0, 11206)	1
  (0, 14526)	1
  (0, 14532)	1


In [128]:
df = pd.DataFrame(X_train_vectorized[0].toarray(), index=['value']).T
df

,value
0,0
1,0
2,0
3,0
4,0
...,...
15722,0
15723,0
15724,0
15725,0


In [129]:
words = list(df[df['value'] > 0].index)
print(words)
[features[index] for index in words]

[185, 590, 1018, 1774, 2470, 2538, 4533, 5016, 5307, 5676, 8053, 9076, 9852, 10109, 11206, 14526, 14532]


['abroad',
 'american',
 'authenticate',
 'british',
 'clich',
 'clumsy',
 'employs',
 'exterior',
 'film',
 'frosty',
 'liability',
 'ms',
 'paltrow',
 'persona',
 'recovers',
 'ugly',
 'ultimately']

#### Train model

In [130]:
clf = LogisticRegression().fit(X_train_vectorized, y_train)
y_pred = clf.predict(X_test_vectorized)
score = clf.decision_function(X_test_vectorized)

def evaluating(y_test, y_pred):
    print(f'Accuracy score {accuracy_score(y_test, y_pred)}')
    print(f'Precision score {precision_score(y_test, y_pred)}')
    print(f'Recall score {recall_score(y_test, y_pred)}')
    print(f'F1 score {f1_score(y_test, y_pred)}')

evaluating(y_test, y_pred)   

Accuracy score 0.7445611402850713
Precision score 0.7467581998474447
Recall score 0.7371987951807228
F1 score 0.7419477074649489


In [131]:
features_array = np.array(features)
sorted_coefs_indices = clf.coef_[0].argsort()

print(f'Smallest coefs: {features_array[sorted_coefs_indices[:10]]}')
print(f'Largest coefs: {features_array[sorted_coefs_indices[:-11:-1]]}')


Smallest coefs: ['dull' 'worst' 'boring' 'tries' 'waste' 'mess' 'mediocre' 'devoid'
 'tedious' 'generic']
Largest coefs: ['solid' 'wonderful' 'engrossing' 'unexpected' 'powerful' 'cinema' 'works'
 'entertaining' 'enjoyable' 'gem']


## TF-IDF

In [132]:
tf_idf_vect = TfidfVectorizer(stop_words='english').fit(X_train)

X_train_vectorized = tf_idf_vect.transform(X_train)
X_test_vectorized = tf_idf_vect.transform(X_test)

#### Train and evaluate model

In [133]:
clf = LogisticRegression().fit(X_train_vectorized, y_train)
y_pred = clf.predict(X_test_vectorized)
score = clf.decision_function(X_test_vectorized)

evaluating(y_test, y_pred)

Accuracy score 0.745311327831958
Precision score 0.7456472369417109
Recall score 0.7417168674698795
F1 score 0.743676859192148


#### Smallest and Largest coefs

In [134]:
features_array = np.array(tf_idf_vect.get_feature_names_out())
sorted_coefs_indices = clf.coef_[0].argsort()

print(f'Smallest coefs: {features_array[sorted_coefs_indices[:10]]}')
print(f'Largest coefs: {features_array[sorted_coefs_indices[:-11:-1]]}')

Smallest coefs: ['bad' 'dull' 'worst' 'boring' 'feels' 'tv' 'tries' 'mess' 'flat' 'thing']
Largest coefs: ['solid' 'cinema' 'entertaining' 'works' 'powerful' 'best' 'heart'
 'performances' 'engrossing' 'enjoyable']


## Word2vect

In [135]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/deructu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [136]:
def tokenize_text(text):
    return [word_tokenize(row) for row in text]

X_train_tokenized = tokenize_text(X_train)
X_test_tokenized = tokenize_text(X_test)

w2v_model = Word2Vec(
    X_train_tokenized,
    vector_size=100,
    window=5,
    min_count=5,
    workers=-1
)

print(f'Vocabulary size: {len(w2v_model.wv.key_to_index)}')

Vocabulary size: 3565


In [137]:
def document_vector(doc, model):
    doc = [word for word in doc if word in model.wv]
    if len(doc) == 0:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[word] for word in doc], axis=0)

X_train_w2v = np.array([document_vector(doc, w2v_model) for doc in X_train_tokenized])
X_test_w2v = np.array([document_vector(doc, w2v_model) for doc in X_test_tokenized])

#### Train NN 

In [85]:
w2v_clf = MLPClassifier(
    hidden_layer_sizes=(64, 64),
    max_iter=2000
)
w2v_clf.fit(X_train_w2v, y_train)

y_pred = w2v_clf.predict(X_test_w2v)
scores = w2v_clf.predict_proba(X_test_w2v)[:, 1]

evaluating(y_test, y_pred)


Accuracy score 0.5671417854463616
Precision score 0.5843023255813954
Recall score 0.45406626506024095
F1 score 0.5110169491525424


## Tranformers

In [ ]:
sample_size = 5000
indices = np.random.choice(len(X_train), sample_size, replace=False)
X_train_sample = X_train.iloc[indices].reset_index(drop=True)
y_train_sample = y_train.iloc[indices].reset_index(drop=True)

train_dataset = Dataset.from_dict({
    'text': X_train_sample,
    'label': y_train_sample
})

test_indices = np.random.choice(len(X_test), min(1000, len(X_test)), replace=False)
X_test_sample = X_test.iloc[test_indices].reset_index(drop=True)
y_test_sample = y_test.iloc[test_indices].reset_index(drop=True)

test_dataset = Dataset.from_dict({
    'text': X_test_sample,
    'label': y_test_sample
})

# Load pre-trained model and tokenizer

In [ ]:
model_name = "distilbert-base-uncased"  # A smaller, faster version of BERT
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize data
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 28343.72it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 1000/1000 [00:00<00:00, 22395.90 examples/s]


#### Training transformer

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="TENSORBOARD_LOGGING_DIR",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/Users/deructu/amazinum/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.687611
20,0.696381
30,0.685693
40,0.682130
50,0.649309
60,0.636865
70,0.540438
80,0.486614
90,0.403420
100,0.549969


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.81it/s]
/Users/deructu/amazinum/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.94it/s]


TrainOutput(global_step=939, training_loss=0.26936987216988056, metrics={'train_runtime': 328.3204, 'train_samples_per_second': 45.687, 'train_steps_per_second': 2.86, 'total_flos': 496752744960000.0, 'train_loss': 0.26936987216988056, 'epoch': 3.0})

#### Evaluate

In [87]:
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)

evaluating(y_test_sample, preds)

/Users/deructu/amazinum/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Accuracy score 0.839
Precision score 0.8290766208251473
Recall score 0.8508064516129032
F1 score 0.8398009950248756


Трансформери показали найкращий результат у класифікації настрою, бо використовують self-attention для захоплення контекстних зв'язків між словами.

Word2Vec дав найгірший результат, бо усереднення векторів слів втрачає частоти  і позицію.

TF-IDF та BoW майже не відрізняються, бо обидва створюють вектори на основі частот слів, але TF-IDF трохи кращий через IDF 